In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# 1. Read the source files
datasets = {
    "products_raw": pd.read_csv("Raw data/source3_products.csv"),
    "business_targets_raw": pd.read_excel(
        "Raw data/source5_business_targets.xlsx"
    ),
    "stores_raw": pd.read_csv("Raw data/source4_stores.tsv", sep="\t"),
    "customers_raw": pd.read_json(
        "Raw data/source2_customers.jsonl", lines=True
    ),
    "sales_raw": pd.read_csv("Raw data/source1_sales.csv"),
}

# 2. Configure the database connection using URL.create
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="0125shavics@",  # Replace with your actual password
    host="127.0.0.1",
    port=5432,
    database="novamart_analytics",
)

engine = create_engine(connection_url)

# 3. Ensure the staging schema exists before loading
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS staging;"))
    conn.commit()

# 4. Load each DataFrame into the PostgreSQL staging schema
for table_name, df in datasets.items():
    print(f"Loading {table_name} into staging schema...")

    # Standardize column headers (lowercase and replace spaces with underscores)
    df.columns = [
        col.strip().lower().replace(" ", "_").replace("-", "_")
        for col in df.columns
    ]

    # Push to SQL
    df.to_sql(
        name=table_name,
        con=engine,
        schema="staging",
        if_exists="replace",  # Overwrites the raw table on every fresh run
        index=False,
        chunksize=10000,  # Inserts in batches for speed and memory safety
        method="multi",  # Multi-row insert for high performance
    )
    print(f" Loaded '{table_name}' with {len(df):,} rows.")

print("\nAll raw tables successfully loaded into the 'staging' schema!")

Loading products_raw into staging schema...
 Loaded 'products_raw' with 150 rows.
Loading business_targets_raw into staging schema...
 Loaded 'business_targets_raw' with 216 rows.
Loading stores_raw into staging schema...
 Loaded 'stores_raw' with 12 rows.
Loading customers_raw into staging schema...
 Loaded 'customers_raw' with 3,050 rows.
Loading sales_raw into staging schema...
 Loaded 'sales_raw' with 25,120 rows.

All raw tables successfully loaded into the 'staging' schema!
